In [12]:
class Curve:
    def __init__(self,coords,ring):
        self.coords=coords
        self.ring=ring
        R = PolynomialRing(QQ, ['x', 'y'])
        self.R=R
        self.r=len(coords)-1
        self.d=coords[0].degree()
    
    def matrix_of_partials(self,k):
        return matrix(R,[[f.derivative(x,i,y,k-i) for i in range(0,k+1)] for f in self.coords])
    
    def all_partials(self,k):
        res={}
        for i in range(1,k+1):
            res[i]=self.matrix_of_partials(i)
        return res
    
    def osc_ideal(self,k):
        M=self.matrix_of_partials(k)
        return R.ideal(M.minors(k+1))
    
    def check_osc_behavior(self,k):
        I=self.osc_ideal(k).radical()
        if I==R.ideal(x,y):
            print('The curve has the correct order',k,'osculating behavior')
            return True
        else:
            print('The curve has the wrong order',k,'osculating behavior')
            return False
    
    def compute_defect(self,k,s):
        Q=[]
        M=self.matrix_of_partials(k)
        for i in range(0,k+1):
            for j in range(0,self.d-k+s+1):
                row=[]
                for l in range(0,self.r+1):
                    for p in range(0,s+1):
                        row.append(M[l][i].coefficient({x:j-p,y:self.d-k+p-j}))
                Q.append(row)
                res=matrix(Q)
        return res
    
    
    def check_balanced(self,k):
        p=((self.d-k)*(k+1))//(self.r-k)
        a=(self.r-k)-((self.d-k)*(k+1))%(self.r-k)
        Q1=self.compute_defect(k,p-1)
        if Q1.ncols()-Q1.rank() != 0:
            print('The matrix of partials has a relation of degree less than',p,'so it is not be balanced.')
        Q2=self.compute_defect(k,p)
        if Q2.ncols()-Q2.rank() != a:
            print('The matrix of partials has too many relations of degree',p,'to be balanced.')
        if Q2.ncols()-Q2.rank() == a and Q1.ncols()-Q1.rank() == 0:
            print('BALANCED')
        
    def get_syzygys(self,k):
        self.check_osc_behavior(k)

        M = transpose(self.matrix_of_partials(k))
    
        # Pass matrix to Singular engine and call .syz()
        S_mat = singular(M)
        syz_sing = S_mat.syz()

        # Convert back to a Sage matrix
        syz_matrix = syz_sing.sage()
        return syz_matrix
            
 

            
    
    

In [37]:
A=curve.get_syzygys(2)
for i in range(0,A.ncols()):
    res=[A[j][i] for j in range(0,A.nrows())]
    print(res)

The curve has the correct order 2 osculating behavior
[-36*y^10, 80*x*y^9, -45*x^2*y^8, x^10, 0, 0, 0, 0]
[0, -80*y^11, 99*x*y^10, -55*x^9*y^2, 36*x^11, 0, 0, 0]
[0, 0, -225*y^11, 4125*x^8*y^3 + 2240*x^4*y^7 - 10800*x^3*y^8 - 784*y^11, -9900*x^10*y - 11200*x^6*y^5 + 24000*x^5*y^6 + 1260*x^2*y^9 - 400*x*y^10, 6000*x^11 + 11200*x^7*y^4, -2240*x^9*y^2 - 19200*x^8*y^3, 1764*x^10*y + 8400*x^9*y^2]
[0, 0, 0, -3360000*x^6*y^5 - 12800000*x^5*y^6 + 192080*x^4*y^7 + 1646400*x^3*y^8 - 9408000*x^2*y^9 - 5600000*x*y^10 - 38467228*y^11, 16800000*x^8*y^3 + 64000000*x^7*y^4 - 960400*x^6*y^5 - 8232000*x^5*y^6 + 20160000*x^4*y^7 + 9600000*x^3*y^8 + 64108045*x^2*y^9 - 34300*x*y^10 - 294000*y^11, -16800000*x^9*y^2 - 64000000*x^8*y^3 + 960400*x^7*y^4 + 8232000*x^6*y^5, 3360000*x^11 + 12800000*x^10*y - 192080*x^9*y^2 - 1646400*x^8*y^3 - 14112000*x^7*y^4, 38551263*x^10*y + 720300*x^9*y^2 + 6174000*x^8*y^3]
[0, 0, 0, 80*x^5*y^6 - 28*x*y^10 + 240*y^11, -400*x^7*y^4 + 45*x^3*y^8 - 400*x^2*y^9, 400*x^8*y^3, -80*

In [36]:
R.<x,y>=PolynomialRing(QQ)
C=[x^20,x^19*y,x^18*y^2,x^10*y^10,x^8*y^12,x^7*y^13+x^2*y^18,x^5*y^15+x*y^19,y^20]
curve=Curve(C,R)
print(curve.coords)
curve.check_balanced(2)

[x^20, x^19*y, x^18*y^2, x^10*y^10, x^8*y^12, x^7*y^13 + x^2*y^18, x^5*y^15 + x*y^19, y^20]
BALANCED
